# EM Algorithm: Multinomial Model & Gaussian Mixture
Two parts: (1) a one-parameter multinomial model and its likelihood, (2) fitting a two-component Gaussian mixture with the EM algorithm (`mixem`).

In [ ]:
import numpy as np
import scipy as sp
import mixem as mm
import random


## 1. Multinomial model with parameter $\theta$
Five categories with cell probabilities
$$
p(\theta) = \left(\tfrac{1}{2},\ \tfrac{\theta}{4},\ \tfrac{1-\theta}{4},\ \tfrac{1-\theta}{4},\ \tfrac{\theta}{4}\right), \qquad \theta \in [0,1]
$$
They sum to 1 for any $\theta$: $\tfrac12 + \tfrac{2\theta}{4} + \tfrac{2(1-\theta)}{4} = 1$.
With $\theta = 1$, categories 3 and 4 have probability 0.

In [ ]:
theta = 1 
p1 = 1/2
p2 = 1/4 * theta 
p3 = 1/4 * (1-theta)
p4 = 1/4 * (1-theta)
p5 = 1/4 * theta

prob = [p1,p2,p3,p4,p5]

### Random sampling demo
- `random.randint(1, 100)`: a uniform integer in $\{1,\dots,100\}$
- `random.random()`: draws from $U[0,1)$

In [ ]:
x = random.randint(1,100)
print(x)

samples = [random.random() for _ in range(10)]
print(samples)

### Multinomial likelihood
For counts $(n_1,\dots,n_5)$ with $n = \sum_i n_i$:
$$
P(n_1,\dots,n_5 \mid \theta) = \frac{n!}{\prod_i n_i!} \prod_{i=1}^{5} p_i^{\,n_i}
$$
Here $n = (10,5,0,0,5)$, $n=20$, $\theta=1$:
$$
P = \frac{20!}{10!\,5!\,5!}\left(\tfrac12\right)^{10}\left(\tfrac14\right)^{10} \approx 0.0434
$$
Dropping constants, the log-likelihood in $\theta$ is
$$
\ell(\theta) = (n_2+n_5)\log\theta + (n_3+n_4)\log(1-\theta)
\;\Rightarrow\;
\hat\theta = \frac{n_2+n_5}{n_2+n_3+n_4+n_5} = \frac{10}{10} = 1
$$
so $\theta=1$ is the MLE for these counts.

In [ ]:
from scipy.stats import multinomial

counts = [10,5,0,0,5]
n=sum(counts)
p = multinomial.pmf(counts, n=n, p=prob)
print("multinomial p:", p)

## 2. EM for a two-component Gaussian mixture
**Data:** 500 points from $\mathcal N(0,1)$ and 500 from $\mathcal N(5,1)$.

**Model:**
$$
p(x) = \sum_{k=1}^{2} \pi_k\, \mathcal N(x \mid \mu_k, \sigma_k^2), \qquad \sum_k \pi_k = 1
$$

**Initialization:** $\mu = (-1, 4)$, $\sigma = (2, 2)$, $\pi = (0.5, 0.5)$.

**E-step:** compute responsibilities (posterior probability that $x_i$ came from component $k$):
$$
\gamma_{ik} = \frac{\pi_k\, \mathcal N(x_i \mid \mu_k, \sigma_k^2)}{\sum_j \pi_j\, \mathcal N(x_i \mid \mu_j, \sigma_j^2)}
$$

**M-step:** with $N_k = \sum_i \gamma_{ik}$, update
$$
\pi_k = \frac{N_k}{N}, \qquad
\mu_k = \frac{1}{N_k}\sum_i \gamma_{ik} x_i, \qquad
\sigma_k^2 = \frac{1}{N_k}\sum_i \gamma_{ik}(x_i-\mu_k)^2
$$

**Repeat** until the log-likelihood stops improving (or 200 iterations):
$$
\log L = \sum_{i=1}^{N} \log \sum_{k} \pi_k\, \mathcal N(x_i \mid \mu_k, \sigma_k^2)
$$
EM never decreases $\log L$, but it can converge to a local maximum, so initialization matters.

In [ ]:
### em algorithm

np.random.seed(42)

data1 = np.concatenate([
np.random.normal(0, 1, 500),
np.random.normal(5, 1, 500)
])

distr = [
    mm.distribution.NormalDistribution(mu= -1, sigma =2),
    mm.distribution.NormalDistribution(mu= 4,sigma=2)
]

weights,distr,log_likelihood = mm.em(
    data1,
    distr,
    initial_weights=[0.5,0.5],
    max_iterations = 200
)


### Results
Expected estimates close to the true values: $\pi \approx (0.5, 0.5)$, $\mu \approx (0, 5)$, $\sigma \approx (1, 1)$.

In [ ]:
print("Estimated mixture weights:", weights)
print("Estimated means:", [d.mu for d in distr])
print("Estimated standard deviations:", [d.sigma for d in distr])
print("Log-likelihood:", log_likelihood)